# 03 - Markov Chain (orders 1-6, Dirichlet-interpolated)

The first model with actual memory: score *"what usually follows the last `o` cities?"* by counting observed transitions in the training trips.

$$P_o(i \mid ctx_o) = \frac{c(ctx_o, i) + \mu \, P_{o-1}(i \mid ctx_{o-1})}{N(ctx_o) + \mu}, \qquad P_0(i)=\text{unigram}$$

**Dirichlet-prior interpolation**: each order's counts are blended with the next-lower order, and $\mu$ sets how strongly. A rarely seen context leans on the lower order; a frequently seen one trusts its own counts. (Add-$k$ smoothing is not used: it adds one constant to every observed count and can never change the ranking.)

No gradient -- just counts. First rung above Popularity (Popularity -> **Markov** -> FPMC -> GRU4Rec -> SASRec). Markov is deterministic: no seeds, no epochs.

### Benchmark protocol (same as `models_benchmarking.ipynb`)
1. **Tune** with seed 42: fit on Train, score on Validation, select on validation **NDCG@10** -- 25 Optuna TPE trials over `max_order` in [1, 6] and `mu` in [0.1, 1000] (log).
2. *(No $E_{best}$: Markov has no epochs.)*
3. **Merge** Train + Validation.
4. **Refit once** (deterministic -- no seeds).
5. **Score on Test** and append the row to `results.csv`.

### Ablation: does the recursion matter?
The same protocol is run twice. **Recursive** (the model above) chains order 6 -> 5 -> ... -> 1 -> unigram. **No recursion** uses only the top-order context, blended straight with the unigram (order 6 -> unigram): if that context is unseen, orders 5-1 are *not* consulted. Same counts, same search space, same seed, same refit -- the recursion is the only difference. Each variant's best configuration is appended to `results.csv` (`Markov` and `Markov (no recursion)`), and a per-order table compares them across all orders 1-6.

All logic is in `src/models/markov.py`. Running this notebook top to bottom does everything and ends by displaying `results.csv`.

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
import os, sys, json
def find_root(start=None):
    d = os.path.abspath(start or os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'src', 'preprocessing.py')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.path.join(os.getcwd(), '..'))
ROOT = find_root(); sys.path.insert(0, ROOT)
import pandas as pd
from src.preprocessing import load_processed, preprocess
from src.evaluate import append_result
try:
    data = load_processed(os.path.join(ROOT, 'data', 'processed'))
except FileNotFoundError:
    data = preprocess(data_dir=os.path.join(ROOT, 'data'), sample_n_trips=5000)
print('loaded:', data['meta'])
if data['meta'].get('sample_n_trips') is not None:
    print('WARNING: processed.pkl was built on a subsample (sample_n_trips=%r). '
          'Delete data/processed/processed.pkl and rebuild with sample_n_trips=None '
          'before trusting or logging any numbers.' % data['meta']['sample_n_trips'])

RESULTS = os.path.join(ROOT, 'results.csv')
TUNING_DIR = os.path.join(ROOT, 'results', 'tuning'); os.makedirs(TUNING_DIR, exist_ok=True)

## Step 1 -- hyper-parameter search (validation only)

Counts are precomputed once up to order 6, so all 25 trials only re-weight them. The test set is never touched here.

In [ ]:
pip install optuna

In [ ]:
from src.models.markov import tune_markov, final_markov, run_markov, N_TRIALS, MODEL_NAME, ABLATION_NAME

VARIANTS = {MODEL_NAME: True, ABLATION_NAME: False}     # name -> recursive?
studies, best = {}, {}
for name, rec in VARIANTS.items():
    print(f"--- {name} (recursive={rec}) ---")
    studies[name] = tune_markov(data, n_trials=N_TRIALS, recursive=rec, show_progress_bar=False)   # 25 trials, TPE seed 42, val NDCG@10
    best[name] = dict(studies[name].best_params)                          # {"max_order": ..., "mu": ...}
    print("selected:", best[name], "| best val NDCG@10 =", round(studies[name].best_value, 4))

best_params = best[MODEL_NAME]                         # kept for the cells below

with open(os.path.join(TUNING_DIR, "markov.json"), "w") as f:
    json.dump({name: {"recursive": VARIANTS[name], "best_params": best[name],
                      "best_val_NDCG@10": studies[name].best_value,
                      "n_trials": len(studies[name].trials)} for name in VARIANTS}, f, indent=2)

for name in VARIANTS:
    print(f"\nTop trials -- {name}")
    trials = studies[name].trials_dataframe()[["number", "value", "params_max_order", "params_mu"]]
    display(trials.sort_values("value", ascending=False).head(10))

## Steps 3-5 -- refit on Train + Validation, score on Test, log both variants to `results.csv`

In [ ]:
for name, rec in VARIANTS.items():
    metrics, _ = final_markov(data, best[name], recursive=rec)   # merges train+val, fits once, scores test
    print(f"TEST [{name}]:", {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})
    append_result(RESULTS, name, metrics)

## The fallback rate -- why Markov plateaus

Whenever a trip's exact top-order context (its last `max_order` cities) was never seen in training, the model leans on a lower order -- or, in the worst case, the unigram prior. `fallback_rate` is the fraction of trips for which that happened. Dirichlet interpolation makes the fallback graceful, but it cannot invent evidence for a context never observed, which is the gap FPMC's factorisation and GRU4Rec/SASRec's learned embeddings are built to close. The table below (train-only fits, validation score; diagnostic, not logged) shows it growing with order.

**It is also the ablation.** For every order 1-6 the table compares the recursive model with the highest-order-only model (each using its own tuned `mu`). At order 1 the two are identical by construction; the gap opens as order grows and the fallback rate rises, because the recursive model recovers lower-order evidence where the ablation drops to the unigram. `delta` = recursive - no recursion (validation NDCG@10).

In [ ]:
rows = []
for order in range(1, 7):
    row = {"order": order}
    for name, rec in VARIANTS.items():
        m, _, val = run_markov(data, order=order, mus=best[name]["mu"], verbose=False, recursive=rec)
        tag = "recursive" if rec else "no_recursion"
        row[f"val_NDCG@10 ({tag})"] = round(val, 4)
        row[f"fallback ({tag})"] = f"{m['fallback_rate']:.1%}"
    row["delta"] = round(row["val_NDCG@10 (recursive)"] - row["val_NDCG@10 (no_recursion)"], 4)
    rows.append(row)
display(pd.DataFrame(rows).set_index("order"))

## Final results (`results.csv`)

In [ ]:
display(pd.read_csv(RESULTS))

## What to write up

- Report the selected (for both variants) `(max_order, mu)` (saved to `results/tuning/markov.json`) together with the **fallback rate**: a high fallback rate is direct evidence for *why* a count table can't generalise the way a learned model can.
- Markov is deterministic, so the final table shows a single run on Train + Val with no +/-.
- `n_params` is the number of distinct `(context, next-city)` counts stored -- the count-based analogue of a neural parameter count.
- Ablation takeaway: compare the `Markov` and `Markov (no recursion)` rows in `results.csv`, and the per-order `delta` column. A positive delta that grows with order means the lower-order backoff is doing real work when the top-order context is sparse.